# Leaf Species Detection

**Goal:** predict the species of an iris flower (setosa, versicolor or virginica) from four measurements of its leaf/petal.

**What you will learn:**
- How to load a CSV dataset with pandas.
- How to explore it (shape, summary, a simple plot).
- How to split data into training and test sets.
- How to train a Decision Tree classifier.
- How to measure accuracy and how tree depth changes it.

**Problem type:** classification (the answer is a category).

**Dataset:** `data/iris.csv` - the classic Iris dataset, 150 flowers, 4 features + a `species` label.

## Import the libraries

- `pandas` loads and inspects tabular data.
- `numpy` does the maths.
- `matplotlib` draws plots.
- `sklearn` holds the machine-learning tools.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

## Load the dataset

The data lives in the `data/` folder, one level up from this notebook. The helper below resolves that relative path whether you launch the notebook from the project root or from inside `notebooks/`.

In [ ]:
def find_data(filename):
    for base in [".", "data", "../data", "../../data"]:
        path = os.path.join(base, filename)
        if os.path.exists(path):
            return path
    raise FileNotFoundError(
        f"Could not find {filename}. Keep the data/ folder next to the notebooks/ folder."
    )


DATA_PATH = find_data("iris.csv")
df = pd.read_csv(DATA_PATH)

print("Loaded:", DATA_PATH)
print("Rows, columns:", df.shape)
df.head()

## Explore the data

Before modelling, always look at your data. How many flowers of each species? What do the numbers look like?

In [ ]:
print("Species counts:")
print(df["species"].value_counts())

print("\nSummary statistics:")
df.describe()

In [ ]:
colors = {"setosa": "red", "versicolor": "green", "virginica": "blue"}

plt.figure(figsize=(7, 5))
for species, group in df.groupby("species"):
    plt.scatter(group["petal_length"], group["petal_width"],
                c=colors[species], label=species)

plt.xlabel("Petal length (cm)")
plt.ylabel("Petal width (cm)")
plt.title("Iris species by petal size")
plt.legend()
plt.show()

You can already see the species form separate clusters - the model just has to learn those boundaries.

## Choose features and target

- **X** = the inputs the model learns from (the four measurements).
- **y** = the answer we want to predict (the numeric `target`: 0, 1 or 2).

In [ ]:
feature_names = ["sepal_length", "sepal_width", "petal_length", "petal_width"]

X = df[feature_names]
y = df["target"]

print("X shape:", X.shape)
print("y shape:", y.shape)

## Split into training and test sets

We train on ~80% of the flowers and keep ~20% hidden to test how well the model generalises to data it has never seen.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training rows:", X_train.shape[0])
print("Test rows:    ", X_test.shape[0])

## Train a Decision Tree

A Decision Tree learns a series of yes/no questions ("is petal length > 2.5?") to split the flowers apart.

In [ ]:
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print(f"Accuracy: {accuracy:.2%}")

## Does tree depth matter?

`max_depth` limits how many questions the tree may ask. Too shallow and it under-learns; too deep and it memorises the training data (overfitting). Let us test a range.

In [ ]:
depths = range(1, 11)
scores = []

for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)
    scores.append(accuracy_score(y_test, tree.predict(X_test)))

plt.figure(figsize=(7, 4))
plt.plot(list(depths), scores, marker="o")
plt.xlabel("max_depth")
plt.ylabel("Test accuracy")
plt.title("Accuracy vs tree depth")
plt.xticks(list(depths))
plt.show()

for depth, score in zip(depths, scores):
    print(f"max_depth={depth}: {score:.2%}")

A small depth (3-4) is usually enough here - a good example of a simpler model doing just as well.

## Predict a brand-new flower

Give the model measurements it has never seen and read off the species. The four values are, in order: sepal length, sepal width, petal length, petal width.

In [ ]:
species_names = ["setosa", "versicolor", "virginica"]
new_flower = pd.DataFrame([[5.1, 3.5, 1.4, 0.2]], columns=feature_names)

prediction = model.predict(new_flower)[0]
print("Predicted species:", species_names[prediction])

## What you learned

- Loading data, choosing **X** and **y**, and splitting train/test.
- Training a classifier with `fit()` and predicting with `predict()`.
- Measuring quality with accuracy and tuning one parameter (`max_depth`).

**Try next:** swap `DecisionTreeClassifier` for `KNeighborsClassifier` or `LogisticRegression` and compare the accuracy.